In [45]:
from PIL import Image
import numpy as np
from deepface import DeepFace
import cv2
import matplotlib.pyplot as plt

In [89]:
# prueba_deepface.py

# --- Cargar imágenes de prueba ---
img1 = "patri2.jpeg"
#img2 = ".\DNI_especimen\dni_front_especimen.jpg"
img2 = '.\dni\dni_anverso.jpg'

# --- Verificación facial ---
resultado = DeepFace.verify(img1, img2,model_name='VGG-Face',detector_backend='retinaface')
print("Verificación:", resultado)

# # --- Análisis rápido de la primera imagen ---
analisis = DeepFace.analyze(img1, actions=['age', 'gender'],detector_backend='retinaface')
print("Análisis facial:", analisis)

Verificación: {'verified': False, 'distance': 0.846355, 'threshold': 0.68, 'confidence': 5.24, 'model': 'VGG-Face', 'detector_backend': 'retinaface', 'similarity_metric': 'cosine', 'facial_areas': {'img1': {'x': 412, 'y': 633, 'w': 296, 'h': 379, 'left_eye': (645, 810), 'right_eye': (521, 758), 'nose': (566, 851), 'mouth_left': (598, 920), 'mouth_right': (477, 876)}, 'img2': {'x': 320, 'y': 888, 'w': 542, 'h': 731, 'left_eye': (727, 1117), 'right_eye': (462, 1110), 'nose': (593, 1284), 'mouth_left': (694, 1420), 'mouth_right': (479, 1415)}}, 'time': 8.35}


Action: gender: 100%|██████████| 2/2 [00:01<00:00,  1.69it/s]

Análisis facial: [{'age': 25, 'region': {'x': 412, 'y': 633, 'w': 296, 'h': 379, 'left_eye': (645, 810), 'right_eye': (521, 758), 'nose': (566, 851), 'mouth_left': (598, 920), 'mouth_right': (477, 876)}, 'face_confidence': 1.0, 'gender': {'Woman': 90.05647301673889, 'Man': 9.943529963493347}, 'dominant_gender': 'Woman'}]


In [ ]:
import cv2
import numpy as np
from deepface import DeepFace
from scipy.spatial.distance import cosine


# CONFIG

MODEL_NAME = "ArcFace"
DETECTOR = "retinaface"
THRESHOLD = 0.68
PROCESS_EVERY_N_FRAMES = 40


#  Extraer embedding del DNI

dni_embedding = DeepFace.represent(
    img_path=".\dni\dni_anverso_recortado.png",
    model_name=MODEL_NAME,
    detector_backend=DETECTOR,
    enforce_detection=True,
)[0]["embedding"]


#  Webcam setup optimizado

cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

frame_count = 0
last_status = "Scanning..."
last_distance = None

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1

    # Procesar solo cada N frames
    if frame_count % PROCESS_EVERY_N_FRAMES == 0:
        try:
            result = DeepFace.represent(
                frame,
                model_name=MODEL_NAME,
                detector_backend=DETECTOR,
                enforce_detection=False,
                anti_spoofing=True
            )

            live_embedding = result[0]["embedding"]
            distance = cosine(dni_embedding, live_embedding)

            last_distance = distance

            if distance < THRESHOLD:
                last_status = "MATCH"
            else:
                last_status = "NO MATCH"

        except Exception as ex:
            last_status = str(ex)

    # Dibujar resultado sin recalcular
    color = (0, 255, 0) if last_status == "MATCH" else (0, 0, 255)

    text = last_status
    if last_distance is not None:
        text += f" | {last_distance:.3f}"

    cv2.putText(
        frame,
        text,
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        color,
        2
    )

    cv2.imshow("Verification", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()